In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q faiss-gpu-cu12 polars numpy

In [ ]:
#========================================
# $2 Configuration
#========================================
import os
import time
import math
import logging
import numpy as np
import polars as pl
import faiss
from dataclasses import dataclass
from tqdm.notebook import tqdm

# Setup logging system
logging.basicConfig(level=logging.INFO, force=True, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class FaissConfig:
    """Hyperparameters configuration for the Vector Index algorithm."""
    version: str           = 'v1'        # Consistency binding for data matrix version
    dim: int               = 384
    index_type: str        = 'IVFFlat'   # Options: 'FlatIP', 'IVFFlat'
    metric: int            = faiss.METRIC_INNER_PRODUCT
    chunk_size: int        = 100_000     # Chunk size to prevent RAM overflow
    max_train_samples: int = 300_000     # Max training sub-sample for K-Means

    # Clear feature flags for population path
    use_gpu_add: bool      = True        # True: add chunks on GPU; False: add on CPU
    gpu_device_id: int     = 0           # GPU id for FAISS operations

    # Progress controls
    show_progress: bool    = True        # Show tqdm progress bar during long-running loops
    log_every_chunks: int  = 5           # Periodic structured log frequency (in chunks)

    # I/O lifecycle management
    base_dir: str          = '/content/drive/My Drive/Thesis/book_recsys'
    path_index_csv: str    = ""
    path_memmap: str       = ""
    path_npy: str          = ""
    path_output_index: str = ""

    def __post_init__(self):
        # Generate target directory maintaining the test environment structure
        out_dir = f"{self.base_dir}/data/processed_2/model_{self.version}"
        os.makedirs(out_dir, exist_ok=True)

        # Link dynamic paths preventing cross-version data mixture conflicts
        self.path_index_csv    = f"{out_dir}/cb_sbert_book_index.csv"
        self.path_memmap       = f"{out_dir}/cb_sbert_matrix.memmap"
        self.path_npy          = f"{out_dir}/cb_sbert_matrix.npy"
        self.path_output_index = f"{out_dir}/cb_sbert_{self.index_type}.index"

In [ ]:
#=================================
# $3 Data Access Layer
#===================================
def count_total_records(csv_path: str) -> int:
    """[SRP] Count total records using Polars Lazy API for O(1) Memory footprint."""
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"[ERROR] CSV Index file not found at: {csv_path}")
    return pl.scan_csv(csv_path).select(pl.len()).collect().item()

def load_vector_source(config: FaissConfig, total_rows: int) -> np.ndarray:
    """[SRP] Provide virtual matrix representation using NumPy Memmap or fallback to NPY."""
    if os.path.exists(config.path_memmap):
        logger.info(f"Out-of-core enabled: Loading memmap file ({total_rows:,} books)")
        return np.memmap(config.path_memmap, dtype='float32', mode='r', shape=(total_rows, config.dim))

    elif os.path.exists(config.path_npy):
        logger.info(f"In-memory enabled: Loading .npy file ({total_rows:,} books)")
        return np.load(config.path_npy).astype('float32')

    raise FileNotFoundError("Vector data (.memmap or .npy) not found")

In [ ]:
#=================================
# $4 Algorithmic Layer
#===================================
def initialize_index(config: FaissConfig, total_rows: int) -> faiss.Index:
    """[SRP] Factory Pattern implementation to construct Vector Space Structure."""
    if config.index_type == 'FlatIP':
        logger.info("Exact Search mode initialized: IndexFlatIP")
        return faiss.IndexFlatIP(config.dim)

    elif config.index_type == 'IVFFlat':
        nlist = int(4 * math.sqrt(total_rows))
        logger.info(f"Approximate Search mode initialized: IndexIVFFlat with {nlist} clusters (K-Means)")
        quantizer = faiss.IndexFlatIP(config.dim)

        # Link native C++ internal tracker to system toggle
        quantizer.verbose = config.show_progress
        index = faiss.IndexIVFFlat(quantizer, config.dim, nlist, config.metric)
        index.verbose = config.show_progress

        return index

    raise ValueError(f"Index algorithm '{config.index_type}' is not supported.")


def train_index_gpu(index: faiss.Index, vectors: np.ndarray, total_rows: int, config: FaissConfig) -> faiss.Index:
    """[SRP] Trigger K-Means clustering using Random Sub-sample mapped to VRAM."""
    if index.is_trained:
        logger.info("Index [FlatIP] bypasses the training step.")
        return index

    logger.info("IVF Classifier requires Training. Proceeding with Random Sampling...")

    if total_rows > config.max_train_samples:
        # Collect random indexes without fully taxing the local RAM
        sample_idx = np.random.choice(total_rows, size=config.max_train_samples, replace=False)
        sample_idx.sort()
        train_data = np.array(vectors[sample_idx], dtype='float32')
    else:
        train_data = np.array(vectors[:], dtype='float32')

    try:
        res = faiss.StandardGpuResources()
        gpu_index = faiss.index_cpu_to_gpu(res, 0, index)

        logger.info("Warming up Data Clusters on GPU T4 Tensor Cores...")
        gpu_index.train(train_data)
        logger.info("GPU Machine Learning Train stage fully completed.")

        cpu_index_trained = faiss.index_gpu_to_cpu(gpu_index)

        del train_data
        return cpu_index_trained

    except (RuntimeError, MemoryError, AttributeError) as e:
        logger.warning(f"GPU train unavailable. Falling back to CPU train: {e}")
        index.train(train_data)
        del train_data
        return index
    except Exception:
        del train_data
        raise

In [ ]:
#=================================
# $5 Population Layer
#===================================
def populate_and_save_index(cpu_index: faiss.Index, vectors: np.ndarray, total_rows: int, config: FaissConfig) -> None:
    """[SRP] Push vector chunks into FAISS with optional GPU add path."""
    logger.info(f"Starting data population engine (Populating {total_rows:,} books by Chunk limit {config.chunk_size:,}).")

    add_index = cpu_index
    gpu_res = None
    add_backend = 'CPU'

    # Optional fast path: move trained CPU index to GPU for chunk add, then move back once.
    if config.use_gpu_add:
        try:
            gpu_res = faiss.StandardGpuResources()
            add_index = faiss.index_cpu_to_gpu(gpu_res, config.gpu_device_id, cpu_index)
            add_backend = 'GPU'
            logger.info("GPU add path enabled by config: chunks will be added on GPU.")
        except (RuntimeError, MemoryError, AttributeError) as e:
            add_index = cpu_index
            add_backend = 'CPU'
            logger.warning(f"GPU add path unavailable, fallback to CPU add: {e}")

    total_chunks = math.ceil(total_rows / config.chunk_size)
    t_start = time.time()

    chunk_iter = range(0, total_rows, config.chunk_size)
    if config.show_progress:
        chunk_iter = tqdm(
            chunk_iter,
            total=total_chunks,
            desc=f'FAISS add ({add_backend})',
            unit='chunk'
        )

    for chunk_idx, i in enumerate(chunk_iter, start=1):
        chunk_data = np.array(vectors[i : i + config.chunk_size], dtype='float32')
        add_index.add(chunk_data)
        del chunk_data

        if config.log_every_chunks > 0 and (chunk_idx % config.log_every_chunks == 0 or chunk_idx == total_chunks):
            elapsed = time.time() - t_start
            chunks_per_sec = chunk_idx / elapsed if elapsed > 0 else 0.0
            done_rows = min(i + config.chunk_size, total_rows)
            logger.info(
                f"Progress: {chunk_idx}/{total_chunks} chunks | rows {done_rows:,}/{total_rows:,} "
                f"| backend={add_backend} | chunks/s={chunks_per_sec:.2f}"
            )

    logger.info("Population process sequence complete.")

    # Persist a CPU index regardless of which add path was used.
    if config.use_gpu_add and add_index is not cpu_index:
        final_index = faiss.index_gpu_to_cpu(add_index)
        del gpu_res
    else:
        final_index = add_index

    faiss.write_index(final_index, config.path_output_index)
    logger.info(f"Successfully saved Binary Index Structure File at: {config.path_output_index}")

In [ ]:
#=================================
# $6 — Orchestrator Main Loop Pipeline
#===================================
def main_build_pipeline(config: FaissConfig):
    start_time = time.time()
    logger.info("=========== FAISS VECTOR PIPELINE INITIALIZED ===========")

    # Step 1. Parse Matrix Dimensions
    total_rows = count_total_records(config.path_index_csv)

    # Step 2. Bind Virtual Memmap Representation Node
    vectors = load_vector_source(config, total_rows)

    # Step 3. Assemble Logical Skeleton Struct
    cpu_index_skeleton = initialize_index(config, total_rows)

    # Step 4. Challenge Multi-Cluster Neural Topology (K-Means)
    trained_index = train_index_gpu(cpu_index_skeleton, vectors, total_rows, config)

    # Step 5. Combine Array Fragments and Export to Storage
    populate_and_save_index(trained_index, vectors, total_rows, config)

    elapsed = time.time() - start_time
    logger.info(f"====== FAISS SYSTEM BOOTSTRAPPED SECURELY - DURATION ELAPSED: {elapsed:.2f}s ======")

# Invoke System Caller
if __name__ == "__main__":
    app_config = FaissConfig()
    main_build_pipeline(app_config)

2026-04-20 07:55:40,977 - =========== FAISS VECTOR PIPELINE INITIALIZED ===========
2026-04-20 07:55:43,105 - In-memory enabled: Loading .npy file (1,173,713 books)
2026-04-20 07:56:19,046 - Approximate Search mode initialized: IndexIVFFlat with 4333 clusters (K-Means)
2026-04-20 07:56:19,116 - IVF Classifier requires Training. Proceeding with Random Sampling...
2026-04-20 07:56:20,120 - Warming up Data Clusters on GPU T4 Tensor Cores...
2026-04-20 07:56:24,988 - GPU Machine Learning Train stage fully completed.
2026-04-20 07:56:25,037 - Starting data population engine (Populating 1,173,713 books by Chunk limit 100,000).
2026-04-20 07:56:25,151 - GPU add path enabled by config: chunks will be added on GPU.


FAISS add (GPU):   0%|          | 0/12 [00:00<?, ?chunk/s]

2026-04-20 07:56:26,634 - Progress: 5/12 chunks | rows 500,000/1,173,713 | backend=GPU | chunks/s=3.37
2026-04-20 07:56:27,908 - Progress: 10/12 chunks | rows 1,000,000/1,173,713 | backend=GPU | chunks/s=3.63
2026-04-20 07:56:28,333 - Progress: 12/12 chunks | rows 1,173,713/1,173,713 | backend=GPU | chunks/s=3.77
2026-04-20 07:56:28,335 - Population process sequence complete.
2026-04-20 07:56:42,226 - Successfully saved Binary Index Structure File at: /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/cb_sbert_IVFFlat.index
2026-04-20 07:56:42,473 - ====== FAISS SYSTEM BOOTSTRAPPED SECURELY - DURATION ELAPSED: 61.50s ======


In [ ]:
#=================================
# $7 — Acceptance Framework
#===================================

def acceptance_test(config: FaissConfig):
    """Validate geometric bounds and structural cohesion of the final exported Index."""
    logger.info("--- Initiating Acceptance Test Routine (Integrity Assurance) ---")

    if not os.path.exists(config.path_output_index):
        logger.error(f"[FAIL] Architecture Error - Missing Index at designated path: {config.path_output_index}")
        return

    idx = faiss.read_index(config.path_output_index)
    actual_total = idx.ntotal
    expected_total = count_total_records(config.path_index_csv)

    if getattr(idx, 'd', None) == config.dim:
        logger.info(f"[PASS] Index dimension check: d={idx.d}")
    else:
        logger.warning(f"[FAIL] Dimension mismatch: index.d={getattr(idx, 'd', 'N/A')} vs config.dim={config.dim}")

    if not idx.is_trained:
        logger.warning("[FAIL] Algorithm Engine bypassed mandatory Training bounds (Trained = False)")
    else:
        logger.info("[PASS] K-Means Cluster Component: Operationally Secured")

    if actual_total == expected_total:
        logger.info(f"[PASS] Vector Volume Metric ({actual_total:,}) perfectly reflects Ground Truth Scale.")
    else:
        logger.warning(f"[FAIL] Point Data Leakage Detected: Rendered Index bounds ({actual_total}) differ from Root Source ({expected_total}).")

    # Top-K smoke search using a small batch of vectors from source
    if expected_total > 0:
        vectors = load_vector_source(config, expected_total)
        query_count = min(5, expected_total)
        top_k = min(10, expected_total)

        query_batch = np.array(vectors[:query_count], dtype='float32')
        scores, indices = idx.search(query_batch, top_k)

        if indices.shape == (query_count, top_k) and scores.shape == (query_count, top_k):
            logger.info(f"[PASS] Top-K search API check: returned shape={indices.shape}")
        else:
            logger.warning(f"[FAIL] Top-K output shape invalid: indices={indices.shape}, scores={scores.shape}")

        invalid_idx = (indices < -1) | (indices >= expected_total)
        if np.any(invalid_idx):
            logger.warning("[FAIL] Top-K contains out-of-range ids.")
        else:
            logger.info("[PASS] Top-K ids are within valid index range.")

# Trigger Post Execution Integrations Check
acceptance_test(FaissConfig())

2026-04-20 07:56:42,534 - --- Initiating Acceptance Test Routine (Integrity Assurance) ---
2026-04-20 07:56:44,837 - [PASS] Index dimension check: d=384
2026-04-20 07:56:44,838 - [PASS] K-Means Cluster Component: Operationally Secured
2026-04-20 07:56:44,841 - [PASS] Vector Volume Metric (1,173,713) perfectly reflects Ground Truth Scale.
2026-04-20 07:56:44,845 - In-memory enabled: Loading .npy file (1,173,713 books)
2026-04-20 07:56:54,946 - [PASS] Top-K search API check: returned shape=(5, 10)
2026-04-20 07:56:54,950 - [PASS] Top-K ids are within valid index range.
